# E-Commerce Customer Churn Prediction

Проект посвящён анализу поведения клиентов e-commerce сервиса и прогнозированию оттока.

Основные этапы:

- загрузка данных;
- проверка качества данных;
- очистка и обработка пропусков;
- EDA;
- feature engineering;
- baseline;
- обучение Logistic Regression и Random Forest;
- сравнение качества моделей;
- интерпретация признаков.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay
)

pd.set_option("display.max_columns", 100)

## 1. Загрузка данных

In [ ]:
DATA_PATH = "../data/ecommerce_customer_churn.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())

## 2. Data Quality Check

In [ ]:
display(df.info())
display(df.describe(include="all").T)

missing = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)
missing["missing_share"] = missing["missing_count"] / len(df)
display(missing.sort_values("missing_count", ascending=False))

print("Duplicates:", df.duplicated().sum())

## 3. EDA

In [ ]:
churn_rate = df["Churn"].mean()
print(f"Churn rate: {churn_rate:.2%}")

df["Churn"].value_counts(normalize=True).sort_index().plot(kind="bar")
plt.title("Churn distribution")
plt.xlabel("Churn")
plt.ylabel("Share")
plt.show()

In [ ]:
churn_by_satisfaction = df.groupby("SatisfactionScore")["Churn"].mean()

churn_by_satisfaction.plot(kind="bar")
plt.title("Churn rate by satisfaction score")
plt.xlabel("Satisfaction score")
plt.ylabel("Churn rate")
plt.show()

display(churn_by_satisfaction.reset_index())

In [ ]:
churn_by_complain = df.groupby("Complain")["Churn"].mean()

churn_by_complain.plot(kind="bar")
plt.title("Churn rate by complaint")
plt.xlabel("Complain")
plt.ylabel("Churn rate")
plt.show()

display(churn_by_complain.reset_index())

In [ ]:
numeric_cols = df.select_dtypes(include=["number"]).columns.drop(["CustomerID", "Churn"])

corr_with_target = (
    df[numeric_cols.tolist() + ["Churn"]]
    .corr(numeric_only=True)["Churn"]
    .sort_values(ascending=False)
)

display(corr_with_target)

## 4. Подготовка данных к моделированию

In [ ]:
target = "Churn"

X = df.drop(columns=[target, "CustomerID"])
y = df[target]

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

## 5. Baseline

In [ ]:
X_test_raw = X_test.copy()

baseline_pred = (
    ((X_test_raw["Complain"] == 1) & (X_test_raw["Tenure"].fillna(X_train["Tenure"].median()) <= 6)) |
    (X_test_raw["SatisfactionScore"] <= 2) |
    (X_test_raw["DaySinceLastOrder"].fillna(X_train["DaySinceLastOrder"].median()) >= 10)
).astype(int)

def get_metrics(name, y_true, y_pred, y_proba=None):
    return {
        "Model": name,
        "ROC-AUC": np.nan if y_proba is None else roc_auc_score(y_true, y_proba),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }

results = []
results.append(get_metrics("Baseline", y_test, baseline_pred))

print(classification_report(y_test, baseline_pred, zero_division=0))

## 6. Logistic Regression

In [ ]:
lr_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

lr_model.fit(X_train, y_train)

lr_proba = lr_model.predict_proba(X_test)[:, 1]
lr_pred = (lr_proba >= 0.5).astype(int)

results.append(get_metrics("Logistic Regression", y_test, lr_pred, lr_proba))

print(classification_report(y_test, lr_pred, zero_division=0))

## 7. Random Forest

In [ ]:
rf_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=250,
        max_depth=8,
        min_samples_leaf=15,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_pred = (rf_proba >= 0.5).astype(int)

results.append(get_metrics("Random Forest", y_test, rf_pred, rf_proba))

print(classification_report(y_test, rf_pred, zero_division=0))

## 8. Сравнение моделей

In [ ]:
metrics_df = pd.DataFrame(results)
display(metrics_df)

metrics_df.to_csv("../reports/model_metrics.csv", index=False)

In [ ]:
RocCurveDisplay.from_predictions(y_test, lr_proba, name="Logistic Regression")
RocCurveDisplay.from_predictions(y_test, rf_proba, name="Random Forest")
plt.title("ROC Curve")
plt.show()

## 9. Feature Importance

In [ ]:
pre = rf_model.named_steps["preprocess"]
rf = rf_model.named_steps["model"]

feature_names = (
    num_cols
    + list(pre.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(cat_cols))
)

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": rf.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

display(feature_importance.head(20))

feature_importance.head(20).to_csv("../reports/feature_importance_top20.csv", index=False)

feature_importance.head(15).sort_values("importance").plot(
    kind="barh",
    x="feature",
    y="importance",
    figsize=(8, 6),
    legend=False
)
plt.title("Top feature importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## 10. Product Recommendations

На основе модели можно предложить следующие продуктовые решения:

- выделять клиентов с высоким churn risk для CRM-кампаний;
- отдельно работать с клиентами, у которых были жалобы;
- отправлять персональные предложения клиентам с большим `DaySinceLastOrder`;
- тестировать cashback и промокоды как инструменты удержания;
- сегментировать клиентов по вероятности оттока и потенциальной ценности.